# 🛍️ ReviewBot 101 — Self-Graded Lab (Simple → Medium)

You're building **ReviewBot**, an assistant that reads product reviews. Along the way you'll practise the Python building blocks of every AI app, then learn three core ways to *talk* to an LLM: **zero-shot**, **few-shot**, and **chain-of-thought** prompting.

| Rank | Score |
|---|---|
| 🐣 Intern | 0–25% |
| 📝 Junior Reviewer | 25–50% |
| 🔧 Bot Builder | 50–70% |
| 💬 Prompt Crafter | 70–88% |
| 🏆 ReviewBot Pro | 88–100% |

**12 tasks · 100 points.** Run each **✅ CHECK** to score. Hints hide in **💡 dropdowns**.


## ⚙️ Setup — run these
The LLM bits are optional live demos; **grading needs no API key**. To run the live demos, add your Groq key in Colab **Secrets** as `groq_api`.

In [ ]:
!pip install -q langchain-groq

In [ ]:
# Optional: set up the live model (only needed for the "Try it live" cells)
try:
    from google.colab import userdata
    from langchain_groq import ChatGroq
    llm = ChatGroq(model='llama-3.3-70b-versatile', temperature=0, api_key=userdata.get('groq_api'))
    print("LLM ready for live demos.")
except Exception as e:
    llm = None
    print("LLM not configured (that's fine — grading still works). Detail:", e)


In [ ]:
# Grader (just run it)
def has_any(text, words):
    t=(text or "").lower(); return any(w in t for w in words)
def has_all(text, words):
    t=(text or "").lower(); return all(w in t for w in words)

class Grader:
    RANKS=[(0.00,"🐣 Intern","Day one on the job."),
           (0.25,"📝 Junior Reviewer","You read the reviews now."),
           (0.50,"🔧 Bot Builder","You wire the pieces together."),
           (0.70,"💬 Prompt Crafter","You speak fluent LLM."),
           (0.88,"🏆 ReviewBot Pro","Reviews fear you.")]
    def __init__(self,total=100): self.s={}; self.m={}; self.total=total
    @property
    def earned(self): return sum(self.s.values())
    def _bar(self,p,w=30): f=int(round(p*w)); return "#"*f+"-"*(w-f)
    def _rank(self):
        p=self.earned/self.total; r=self.RANKS[0]
        for t,n,b in self.RANKS:
            if p>=t: r=(t,n,b)
        return r
    def check(self,tid,passed,pts,ok="",fail=""):
        self.m[tid]=pts; done=self.s.get(tid,0)==pts
        if passed:
            self.s[tid]=pts
            print(("Already done" if done else "PASS  +"+str(pts)+" pts")+"  [Task "+tid+"]")
            if ok and not done: print("   "+ok)
        else:
            if not done: self.s[tid]=0
            print("Not yet  (0/"+str(pts)+")  [Task "+tid+"]")
            if fail: print("   "+fail)
        p=self.earned/self.total; _,n,b=self._rank()
        print(); print("   "+str(self.earned)+"/"+str(self.total)+"  ["+self._bar(p)+"]  "+str(round(p*100))+"%")
        print("   "+n+" - "+b); print()
    def scorecard(self):
        p=self.earned/self.total; _,n,b=self._rank()
        print("="*50); print("  FINAL SCORECARD"); print("="*50)
        for tid in sorted(self.m,key=lambda x:(len(x),x)):
            e,t=self.s.get(tid,0),self.m[tid]
            print("  "+("[x]" if e==t else ("[~]" if e>0 else "[ ]"))+"  Task "+tid.ljust(4)+" "+str(e).rjust(3)+"/"+str(t))
        print("  "+"-"*40); print("  TOTAL  "+str(self.earned).rjust(3)+"/"+str(self.total)+"  ["+self._bar(p)+"]  "+str(round(p*100))+"%")
        print("  "+n+"\n  "+b); print("="*50)
grader=Grader(100)
print("Grader ready. 100 points, 12 tasks.")


---
## TASK 1 — Bot settings &nbsp;`5 pts`

Create three variables: **`bot_name`** = `"ReviewBot"` (string), **`model_version`** = `3.5` (float), **`is_live`** = `True` (boolean). Then `print(bot_name)`.


In [ ]:
# TASK 1


<details><summary>💡 Hint</summary>

```python
bot_name = "ReviewBot"
model_version = 3.5
is_live = True
print(bot_name)
```
</details>

In [ ]:
# ✅ CHECK 1
try:
    ok = bot_name=="ReviewBot" and isinstance(model_version,float) and model_version==3.5 and is_live is True
    grader.check("1", ok, 5, ok="Variables set with correct types.",
        fail="bot_name="+repr(globals().get('bot_name'))+", model_version="+repr(globals().get('model_version'))+", is_live="+repr(globals().get('is_live')))
except NameError as e:
    grader.check("1", False, 5, fail="Missing: "+str(e))


---
## TASK 2 — A prompt template &nbsp;`5 pts`

Given `product = "wireless earbuds"`, use an **f-string** to build a prompt and store it in **`prompt`**:
> `"Write a short, honest review of the wireless earbuds."`


In [ ]:
# TASK 2
product = "wireless earbuds"


<details><summary>💡 Hint</summary>

```python
prompt = f"Write a short, honest review of the {product}."
```
</details>

In [ ]:
# ✅ CHECK 2
try:
    ok = "wireless earbuds" in prompt and "review" in prompt.lower()
    grader.check("2", ok, 5, ok='prompt = "'+prompt+'"',
        fail="prompt should mention the product and ask for a review. Got: "+repr(globals().get('prompt')))
except NameError:
    grader.check("2", False, 5, fail="`prompt` not defined.")


---
## TASK 3 — Rating router &nbsp;`8 pts`

A review has a star rating `stars = 2`. Set **`category`**:
- `stars >= 4` → `"Promote"`
- `stars == 3` → `"Monitor"`
- otherwise → `"Escalate"`


In [ ]:
# TASK 3
stars = 2


<details><summary>💡 Hint</summary>

```python
if stars >= 4:
    category = "Promote"
elif stars == 3:
    category = "Monitor"
else:
    category = "Escalate"
```
</details>

In [ ]:
# ✅ CHECK 3
try:
    ok = category == "Escalate"
    grader.check("3", ok, 8, ok='category = "'+category+'" (2 stars -> Escalate).',
        fail="With stars=2 you should get 'Escalate'. Got "+repr(globals().get('category')))
except NameError:
    grader.check("3", False, 8, fail="`category` not defined.")


---
## TASK 4 — Route every review &nbsp;`10 pts`

Loop through `reviews` and, using the `in` keyword, **append** the right team to **`routes`**:
- contains `"crash"` → `"Bug Team"`
- contains `"charged"` → `"Billing Team"`
- otherwise → `"General"`

Start with an empty list `routes = []`.


In [ ]:
# TASK 4
reviews = ["The app keeps crashing on launch",
           "I was charged twice this month",
           "Honestly love the new design"]


<details><summary>💡 Hint</summary>

```python
routes = []
for r in reviews:
    if "crash" in r:
        routes.append("Bug Team")
    elif "charged" in r:
        routes.append("Billing Team")
    else:
        routes.append("General")
```
</details>

In [ ]:
# ✅ CHECK 4
try:
    ok = routes == ["Bug Team","Billing Team","General"]
    grader.check("4", ok, 10, ok="Routed correctly: "+str(routes),
        fail="Expected ['Bug Team','Billing Team','General']. Got "+repr(globals().get('routes')))
except NameError:
    grader.check("4", False, 10, fail="`routes` not defined.")


---
## TASK 5 — Build the tool list &nbsp;`6 pts`

Start with `bot_tools = ["sentiment"]`. **Append** `"summary"` then `"translate"`. Then store the **last** tool (use negative indexing) in **`last_tool`**.


In [ ]:
# TASK 5


<details><summary>💡 Hint</summary>

```python
bot_tools = ["sentiment"]
bot_tools.append("summary")
bot_tools.append("translate")
last_tool = bot_tools[-1]
```
</details>

In [ ]:
# ✅ CHECK 5
try:
    ok = bot_tools==["sentiment","summary","translate"] and last_tool=="translate"
    grader.check("5", ok, 6, ok="bot_tools="+str(bot_tools)+", last_tool='"+last_tool+"'",
        fail="bot_tools="+repr(globals().get('bot_tools'))+", last_tool="+repr(globals().get('last_tool')))
except NameError as e:
    grader.check("5", False, 6, fail="Missing: "+str(e))


---
## TASK 6 — Config dictionary &nbsp;`8 pts`

Create a dictionary **`config`** with `"model": "llama-3"`, `"temperature": 0.2`, `"max_tokens": 300`. Then build **`config_line`** using an f-string that reads *from the dictionary*:
> `"Running llama-3 at temperature 0.2"`


In [ ]:
# TASK 6


<details><summary>💡 Hint</summary>

Use double quotes outside, single quotes for the keys inside:
```python
config = {"model": "llama-3", "temperature": 0.2, "max_tokens": 300}
config_line = f"Running {config['model']} at temperature {config['temperature']}"
```
</details>

In [ ]:
# ✅ CHECK 6
try:
    ok = config.get("model")=="llama-3" and config.get("temperature")==0.2 and "llama-3" in config_line and "0.2" in config_line
    grader.check("6", ok, 8, ok='config_line = "'+config_line+'"',
        fail="config="+repr(globals().get('config'))+", config_line="+repr(globals().get('config_line')))
except NameError as e:
    grader.check("6", False, 8, fail="Missing: "+str(e))


---
## TASK 7 — Print the conversation &nbsp;`8 pts`

Loop through `chat` (a list of dictionaries) and **append** a formatted line to **`transcript`** in the form `"[role]: content"`.


In [ ]:
# TASK 7
chat = [
    {"role": "user", "content": "Is this product waterproof?"},
    {"role": "bot",  "content": "Yes, it is rated IP68."},
    {"role": "user", "content": "Great, thanks!"}
]


<details><summary>💡 Hint</summary>

```python
transcript = []
for msg in chat:
    transcript.append(f"[{msg['role']}]: {msg['content']}")
```
</details>

In [ ]:
# ✅ CHECK 7
try:
    exp = ["[user]: Is this product waterproof?","[bot]: Yes, it is rated IP68.","[user]: Great, thanks!"]
    grader.check("7", transcript==exp, 8, ok="Transcript formatted correctly.",
        fail="Expected "+str(exp)+"\n   Got "+repr(globals().get('transcript')))
except NameError:
    grader.check("7", False, 8, fail="`transcript` not defined.")


---
## TASK 8 — Parse the API response &nbsp;`8 pts`

Extract just the assistant's text from this LLM API response and store it in **`reply`**. Also extract the `finish_reason` into **`reason`**.


In [ ]:
# TASK 8
api_response = {
    "id": "chat-789",
    "usage": {"prompt_tokens": 40, "completion_tokens": 12},
    "choices": [
        {"index": 0,
         "message": {"role": "assistant", "content": "These earbuds have great battery life."},
         "finish_reason": "stop"}
    ]
}


<details><summary>💡 Hint</summary>

```python
reply = api_response['choices'][0]['message']['content']
reason = api_response['choices'][0]['finish_reason']
```
</details>

In [ ]:
# ✅ CHECK 8
try:
    ok = reply=="These earbuds have great battery life." and reason=="stop"
    grader.check("8", ok, 8, ok='reply="'+reply+'", reason="'+reason+'"',
        fail="reply="+repr(globals().get('reply'))+", reason="+repr(globals().get('reason')))
except NameError as e:
    grader.check("8", False, 8, fail="Missing: "+str(e))


---
## TASK 9 — Tokenise by splitting &nbsp;`6 pts`

The cost of an LLM call depends on the number of tokens. As a rough check, split `sentence` into words and store the list in **`tokens`** and the count in **`token_count`**.


In [ ]:
# TASK 9
sentence = "this product exceeded all of my expectations"


<details><summary>💡 Hint</summary>

```python
tokens = sentence.split()
token_count = len(tokens)
```
</details>

In [ ]:
# ✅ CHECK 9
try:
    ok = tokens==sentence.split() and token_count==7
    grader.check("9", ok, 6, ok="Got "+str(token_count)+" tokens.",
        fail="tokens="+repr(globals().get('tokens'))+", token_count="+repr(globals().get('token_count')))
except NameError as e:
    grader.check("9", False, 6, fail="Missing: "+str(e))


---
# 💬 Three ways to prompt an LLM

You already know how to build a prompt with an f-string and send it with `llm.invoke(...)`. *How* you word that prompt changes the answer a lot. Three classics:

- **Zero-shot** — just ask, no examples. *"Classify the sentiment of this review."*
- **Few-shot** — show a few worked examples first, then ask. The model copies the pattern.
- **Chain-of-thought** — ask the model to **reason step by step** before answering. Great for anything that needs working-out.


---
## TASK 10 — Zero-shot prompt &nbsp;`10 pts`

Build **`zero_shot_prompt`**: directly ask the model to classify `new_review` as Positive, Negative, or Neutral — **with no examples**. Make sure the review text is inside the prompt (use an f-string).


In [ ]:
# TASK 10
new_review = "The screen is gorgeous but the battery dies fast."


<details><summary>💡 Hint</summary>

```python
zero_shot_prompt = f"Classify the sentiment of this review as Positive, Negative, or Neutral.\nReview: {new_review}\nSentiment:"
```
</details>

In [ ]:
# ✅ CHECK 10
try:
    ok = (new_review in zero_shot_prompt
          and has_any(zero_shot_prompt, ["sentiment","classify","positive","negative","neutral"])
          and len(zero_shot_prompt) > 40)
    grader.check("10", ok, 10, ok="Zero-shot prompt asks the task directly and includes the review.",
        fail="Include the review text AND an instruction (classify/sentiment...). Got: "+repr(globals().get('zero_shot_prompt'))[:120])
except NameError:
    grader.check("10", False, 10, fail="`zero_shot_prompt` not defined.")


### ▶️ Try it live *(optional)*

In [ ]:
try:
    print(llm.invoke(zero_shot_prompt).content)
except Exception as e:
    print("Add your API key to run this. Detail:", e)


---
## TASK 11 — Few-shot prompt &nbsp;`14 pts`

Now *teach by example*. Using the `examples` list of dictionaries, **build** `few_shot_prompt` by **looping** over the examples and adding each as `Review: ... / Sentiment: ...`, then finish with the new review and a trailing `Sentiment:` for the model to complete.


In [ ]:
# TASK 11
examples = [
    {"review": "Absolutely fantastic, worth every penny!", "label": "Positive"},
    {"review": "Stopped working after two days. Useless.",   "label": "Negative"},
    {"review": "It does the job, nothing more.",             "label": "Neutral"},
]
new_review = "Setup was confusing but it works well now."


<details><summary>💡 Hint</summary>

Start the prompt, loop the examples, then append the query — all with f-strings (string `+=` adds on):
```python
few_shot_prompt = "Classify each review as Positive, Negative, or Neutral.\n\n"
for ex in examples:
    few_shot_prompt += f"Review: {ex['review']}\nSentiment: {ex['label']}\n\n"
few_shot_prompt += f"Review: {new_review}\nSentiment:"
```
</details>

In [ ]:
# ✅ CHECK 11
try:
    p = few_shot_prompt
    has_labels = has_all(p, ["positive","negative","neutral"])
    has_examples = all(ex["review"] in p for ex in examples)
    has_query = new_review in p
    grader.check("11", has_labels and has_examples and has_query, 14,
        ok="Few-shot prompt includes all 3 examples and the new review.",
        fail="labels_ok="+str(has_labels)+", examples_in="+str(has_examples)+", query_in="+str(has_query))
except NameError:
    grader.check("11", False, 14, fail="`few_shot_prompt` not defined.")


### ▶️ Try it live *(optional)*

In [ ]:
try:
    print(llm.invoke(few_shot_prompt).content)
except Exception as e:
    print("Add your API key to run this. Detail:", e)


---
## TASK 12 — Chain-of-thought prompt &nbsp;`12 pts`

Some questions need working-out. Build **`cot_prompt`** that asks the model to **reason step by step** before giving the final answer to `problem`.


In [ ]:
# TASK 12
problem = "A pack has 3 earbuds sets. Each set costs 1499 rupees. With a 10% discount, what is the total?"


<details><summary>💡 Hint</summary>

```python
cot_prompt = f"{problem}\nLet's think step by step, then give the final answer."
```
</details>

In [ ]:
# ✅ CHECK 12
try:
    ok = (problem in cot_prompt
          and (has_any(cot_prompt, ["step by step","step-by-step"]) or has_all(cot_prompt, ["step","reason"]) or has_all(cot_prompt,["think","step"]))
          and len(cot_prompt) > len(problem))
    grader.check("12", ok, 12, ok="Chain-of-thought prompt asks for step-by-step reasoning.",
        fail="Include the problem AND ask it to reason step by step. Got: "+repr(globals().get('cot_prompt'))[:140])
except NameError:
    grader.check("12", False, 12, fail="`cot_prompt` not defined.")


### ▶️ Try it live *(optional)* — compare with and without 'step by step'

In [ ]:
try:
    print("WITH chain-of-thought:\n", llm.invoke(cot_prompt).content)
except Exception as e:
    print("Add your API key to run this. Detail:", e)


---
## 🏆 Final Scorecard

In [ ]:
grader.scorecard()

## 🎓 Recap
You built the Python plumbing of an AI app — variables, prompts, routing, loops, lists, dictionaries, API parsing, tokenising — and three ways to prompt an LLM: **zero-shot** (just ask), **few-shot** (show examples), and **chain-of-thought** (reason step by step). Next stop: the medium-to-hard NLP notebook.
